###This notebook contains the charts discussed in the *Analysis and discussion* chapter of the thesis, along with code for reproducing them in an interactive format.

### It is organized into several sections that follow the sequence of chart descriptions in the thesis (expect for *Helpers* section that contains  functions used for loading JSON files, processing chart data, and building interactive visualizations).

# Helpers

In [1]:
import json
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
DATA_DIR = Path("/content/")

In [3]:
def load_chart_json(filename):
    path = DATA_DIR / filename
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [100]:
def compute_plot_width(n_points, min_width=1000, per_point=55, max_width=5000):
    width = max(min_width, n_points * per_point)
    return min(width, max_width)

In [99]:
def plot_chart_from_json(
    filename,
    title=None,
    min_bar_display=None,
):
    obj = load_chart_json(filename)
    chart_type = obj.get("chart_type")
    graph_mode = obj.get("graph_mode")

    if title is None:
        title = filename.replace(".json", "").replace("_", " ")

    if graph_mode == "distribution":
        df = pd.DataFrame(obj["data"]).copy()

        if "count" not in df.columns:
            raise ValueError(f"{filename}: expected 'count' column for distribution chart")

        x_col = "label" if "label" in df.columns else df.columns[0]

        df["real_count"] = df["count"]

        if min_bar_display is not None:
            df["display_count"] = df["real_count"].apply(
                lambda x: max(x, min_bar_display) if x > 0 else 0
            )
        else:
            df["display_count"] = df["real_count"]

        width = compute_plot_width(len(df), min_width=1400, per_point=70, max_width=7000)

        fig = px.bar(
            df,
            x=x_col,
            y="display_count",
            title=title
        )

        fig.update_layout(
            width=width,
            xaxis_title="Metric value",
            yaxis_title="Count",
            dragmode="pan",
        )

        fig.update_traces(
            customdata=df[["real_count"]],
            hovertemplate="<b>%{x}</b><br>count: %{customdata[0]}<extra></extra>"
        )

        return fig, obj, df

    elif chart_type == "bar":
        df = pd.DataFrame(obj["data"]).copy()

        if "group" in df.columns:
            x_col = "group"
        elif "label" in df.columns:
            x_col = "label"
        else:
            x_col = df.columns[0]

        y_col = "mean" if "mean" in df.columns else "value"

        hover_data = {}
        if "count" in df.columns:
            hover_data["count"] = True
        if "documents_count" in df.columns:
            hover_data["documents_count"] = True
        if "year_min" in df.columns:
            hover_data["year_min"] = True
        if "year_max" in df.columns:
            hover_data["year_max"] = True

        width = compute_plot_width(len(df), min_width=2200, per_point=100, max_width=14000)

        fig = px.bar(
            df,
            x=x_col,
            y=y_col,
            title=title,
           hover_data=hover_data
        )

        fig.update_layout(
            width=width,
            xaxis_title="",
            yaxis_title="Mean value",
            dragmode="pan"
        )

        fig.update_xaxes(
            type="category",
            tickmode="array",
            tickvals=df[x_col].tolist(),
            ticktext=df[x_col].tolist(),
            tickangle=-40,
            automargin=True,
            categoryorder="array",
            categoryarray=df[x_col].tolist()
        )

        fig.update_traces(
            hovertemplate="<b>%{x}</b><br>mean: %{y:.2f}<extra></extra>"
        )

        return fig, obj, df

    elif chart_type == "line":
        df = pd.DataFrame(obj["data"]).copy()

        x_col = "period" if "period" in df.columns else df.columns[0]
        y_col = "mean" if "mean" in df.columns else "value"

        if "decade" in df.columns:
            df = df.sort_values("decade")

        width = compute_plot_width(len(df), min_width=1200, per_point=90, max_width=4000)

        hover_data = {}
        if "count" in df.columns:
            hover_data["count"] = True
        if "documents_count" in df.columns:
            hover_data["documents_count"] = True

        fig = px.line(
            df,
            x=x_col,
            y=y_col,
            markers=True,
            title=title,
            hover_data=hover_data,
        )

        fig.update_layout(
            width=width,
            xaxis_title="Decade",
            yaxis_title="Mean value",
            dragmode="pan",
        )

        fig.update_traces(
            hovertemplate="<b>%{x}</b><br>mean: %{y:.2f}<extra></extra>"
        )

        return fig, obj, df

    elif chart_type == "entity_timeline":
        line_df = pd.DataFrame(obj["data"]["line"]).copy()
        points_df = pd.DataFrame(obj["data"]["points"]).copy()

        if "decade" in line_df.columns:
            line_df = line_df.sort_values("decade")

        width = compute_plot_width(len(line_df), min_width=1200, per_point=90, max_width=4000)

        fig = go.Figure()

        line_x = "period" if "period" in line_df.columns else line_df.columns[0]
        line_y = "mean" if "mean" in line_df.columns else line_df.columns[1]

        fig.add_trace(go.Scatter(
            x=line_df[line_x],
            y=line_df[line_y],
            mode="lines+markers",
            name="Overall trend",
            hovertemplate="<b>%{x}</b><br>mean: %{y:.2f}<extra></extra>"
        ))

        name_col = "entity_name"

        x_col = "period" if "period" in points_df.columns else (
            "decade_label" if "decade_label" in points_df.columns else points_df.columns[0]
        )
        y_col = "mean" if "mean" in points_df.columns else "value"

        trace_kwargs = dict(
            x=points_df[x_col],
            y=points_df[y_col],
            mode="markers",
            name=obj.get("entity_type", "entities").title()
        )

        if name_col is not None:
            trace_kwargs["text"] = points_df[name_col]
            trace_kwargs["hovertemplate"] = (
                "<b>%{text}</b><br>"
                f"{x_col}: %{{x}}<br>"
                f"{y_col}: %{{y:.2f}}<extra></extra>"
            )

        fig.add_trace(go.Scatter(**trace_kwargs))

        fig.update_layout(
            title=title,
            width=width,
            xaxis_title="Decade",
            yaxis_title="Mean value",
            dragmode="pan",
        )

        return fig, obj, {"line": line_df, "points": points_df}

    else:
        raise ValueError(f"Unsupported chart_type: {chart_type}")

#General distributions

In [84]:
fig, obj, df = plot_chart_from_json(
    "GENERAL_sent_len.json",
    "Distribution of sentence length",
    min_bar_display=2000
)
fig.show()

In [49]:
fig, obj, df = plot_chart_from_json(
    "GENERAL_mean_dep_len.json",
    "Distribution of mean dependency length",
    min_bar_display=15000
)
fig.show()

In [56]:
fig, obj, df = plot_chart_from_json(
    "GENERAL_max_tree_dep.json",
    "Distribution of maximum tree depth",
    min_bar_display=5000
)
fig.show()

In [53]:
fig, obj, df = plot_chart_from_json(
    "GENERAL_ratio_subj_clauses.json",
    "Distribution of the proportion of subordinate dependency relations",
    min_bar_display=15000
)
fig.show()

# Authors

In [76]:
fig, obj, df = plot_chart_from_json(
    "AUTHORS_sent_len.json",
    "Distribution of sentence length"
)
fig.show()

In [77]:
fig, obj, df = plot_chart_from_json(
    "AUTHORS_mean_dep_len.json",
    "Distribution of mean dependency length"
)
fig.show()

In [78]:
fig, obj, df = plot_chart_from_json(
    "AUTHORS_max_tree_dep.json",
    "Distribution of maximum tree depth"
)
fig.show()

In [79]:
fig, obj, df = plot_chart_from_json(
    "AUTHORS_ratio_subj_clauses.json",
    "Distribution of the proportion of subordinate dependency relations"
)
fig.show()

# Genres

In [80]:
fig, obj, df = plot_chart_from_json(
    "GENRES_sent_len.json",
    "Distribution of sentence length"
)
fig.show()

In [81]:
fig, obj, df = plot_chart_from_json(
    "GENRES_max_tree_dep.json",
    "Distribution of maximum tree depth"
)
fig.show()

In [82]:
fig, obj, df = plot_chart_from_json(
    "GENRES_ratio_subj_clauses.json",
    "Distribution of the proportion of subordinate dependency relations"
)
fig.show()

In [83]:
fig, obj, df = plot_chart_from_json(
    "GENRES_ratio_conj.json",
    "Distribution of the proportion of conjunction relations"
)
fig.show()

# Temporal dynamics

In [88]:
fig, obj, df = plot_chart_from_json(
    "DECADES_sent_len.json",
    "Distribution of sentence length"
)
fig.show()

In [89]:
fig, obj, df = plot_chart_from_json(
    "DECADES_mean_dep_len.json",
    "Distribution of mean dependency length"
)
fig.show()

In [91]:
fig, obj, df = plot_chart_from_json(
    "DECADES_max_tree_dep.json",
    "Distribution of maximum tree depth"
)
fig.show()

# Objects over time

In [96]:
fig, obj, df = plot_chart_from_json(
    "OVERTIME_AUTHORS_sent_len.json",
    "Distribution of maximum tree depth"
)
fig.show()

In [97]:
fig, obj, df = plot_chart_from_json(
    "OVERTIME_GENRES_sent_len.json",
    "Distribution of maximum tree depth"
)
fig.show()